# 02 QLoRA 학습 (HyperCLOVA X SEED 14B Think)

| 항목 | 값 |
|---|---|
| GPU | A100 40GB 또는 80GB (둘 다 가능) |
| 예상 시간 | 약 3~3.5시간 (다운로드 15분, 학습률 2개 × 2 epoch 학습 2.5시간, 검증 생성 4회 30분) |
| 예상 컴퓨트 | 약 40~45 CU (A100 약 13 CU/시간 가정. Colab 리소스 창에서 실제 소모율을 확인해 기록) |
| 디스크 | 학생 원본 float32 59GB |
| 이어 하기 | Trainer 체크포인트(epoch마다)를 Drive `lora/`에 저장하고, 다시 실행하면 마지막 체크포인트에서 이어 한다. 검증 생성도 이어 쓴다. |

사전 등록: `docs/spec.md` §6.1~6.3, `docs/decisions.md` 2026-11-10. 설정을 바꾸면 사전 등록 위반이니 멈추고 기록한다.
준비: repo에서 `scripts/pack_distill_bundle.sh`로 만든 `tomato_distill_bundle.tar.gz`를 Drive `MyDrive/tomato-distill/`에 올린다.

In [ ]:
# 공통 준비: Drive 연결, 묶음 풀기, 공용 코드 import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, time, gc
DIR = "/content/drive/MyDrive/tomato-distill"
os.environ["TOMATO_DISTILL_DIR"] = DIR
# 묶음은 실행할 때마다 새로 푼다(예전 묶음이 남아 다른 프롬프트로 도는 것을 막는다). sha256을 repo에서 만든 값과 비교한다
import shutil, hashlib
shutil.rmtree(f"{DIR}/bundle", ignore_errors=True)
os.makedirs(f"{DIR}/bundle")
!tar xzf {DIR}/tomato_distill_bundle.tar.gz -C {DIR}/bundle
with open(f"{DIR}/tomato_distill_bundle.tar.gz", "rb") as f:
    print("bundle sha256", hashlib.sha256(f.read()).hexdigest())
sys.path.insert(0, f"{DIR}/bundle/notebooks")
import distill_common as dc
prompt = dc.setup()
from tomato.explain import grounding
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers==5.22.2" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0"
import importlib.metadata as _md
PINS = {'transformers': '5.22.2', 'peft': '0.21.2', 'bitsandbytes': '0.50.2', 'accelerate': '1.15.0'}
assert {k: _md.version(k) for k in PINS} == PINS, {k: _md.version(k) for k in PINS}
print(PINS)
import torch, transformers, peft, bitsandbytes

## SMOKE: 본 실행 전에 전 과정을 작게 확인
- `SMOKE = True`: 학습 32개·검증 8개, 학습률 1e-4 하나로 학습(2 epoch, epoch당 2 step) → 체크포인트 → 검증 생성 → 근거 검사 → 설정 선택 → `chosen` 저장까지 돈다. 결과는 Drive `lora_smoke/`에 쓰고 본 실행의 `lora/`는 건드리지 않는다. 학생 원본 다운로드(첫 실행 약 15분) 뒤 모델 로드 3번과 생성으로 수 분이 걸린다.
- 확인이 끝나면 `SMOKE = False`로 바꾸고 처음부터 다시 실행한다. 학습 설정(spec §6.2)은 SMOKE와 본 실행이 같고, 데이터 수와 학습률 개수만 다르다.

In [ ]:
SMOKE = True  # 본 실행은 False
LORA_DIR = dc.DRIVE / ("lora_smoke" if SMOKE else "lora")
print("SMOKE" if SMOKE else "본 실행", LORA_DIR)

## 데이터: 선생 출력 중 근거 검사·형식 통과분 (01에서 만든 `*_final.jsonl`)

In [ ]:
train_in, val_in = dc.subsets()
inputs = {r["id"]: r["input"] for r in train_in + val_in}
train_t = dc.read_jsonl(dc.DRIVE / "teacher" / "train_final.jsonl")
val_t = dc.read_jsonl(dc.DRIVE / "teacher" / "val_final.jsonl")
if SMOKE:
    train_t, val_in = train_t[:32], val_in[:8]
    val_t = [r for r in val_t if r["id"] in {x["id"] for x in val_in}]
print(len(train_t), len(val_t), len(val_in))
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV)
tok.padding_side = "right"
train_ds = [dc.sft_features(tok, prompt, inputs[r["id"]], r["text"]) for r in train_t]
val_ds = [dc.sft_features(tok, prompt, inputs[r["id"]], r["text"]) for r in val_t]
print("최대 토큰", max(len(x["input_ids"]) for x in train_ds))
print(tok.decode(train_ds[0]["input_ids"])[-600:])

## 학습: r 16, alpha 32, 학습률 {1e-4, 2e-4} × epoch {1, 2} (epoch마다 체크포인트)

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import glob, math

def load_base():
    m = AutoModelForCausalLM.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV, quantization_config=dc.bnb_config(),
                                             dtype=torch.bfloat16, device_map={"": 0})
    return prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)

G = dc.LORA_GRID
LRS = G["lr"][:1] if SMOKE else G["lr"]
for lr in LRS:
    out = LORA_DIR / f"r{G['r']}_lr{lr:g}"
    if (out / "done").exists():
        continue
    model = get_peft_model(load_base(), LoraConfig(r=G["r"], lora_alpha=G["alpha"], lora_dropout=G["dropout"],
                                                   target_modules=G["targets"], task_type="CAUSAL_LM"))
    args = TrainingArguments(output_dir=str(out), num_train_epochs=G["epochs"], learning_rate=lr,
                             per_device_train_batch_size=4, gradient_accumulation_steps=4, per_device_eval_batch_size=8,
                             lr_scheduler_type="cosine", warmup_steps=0.03, bf16=True, logging_steps=10,
                             eval_strategy="epoch", save_strategy="epoch", save_total_limit=None, seed=dc.SEED,
                             gradient_checkpointing=True, optim="paged_adamw_8bit", report_to="none")
    # warmup 3%(spec §6.2): transformers 5.x는 warmup_ratio를 없앴고, warmup_steps에 1보다 작은 소수를 주면
    # 전체 step × 0.03을 올림한 step 수로 쓴다(TrainingArguments.get_warmup_steps, 예전 warmup_ratio와 같은 계산).
    # 전체 step 계산은 Trainer.set_initial_training_values와 같다(batch 수 → 누적 단위 올림 → × epoch).
    n_batches = math.ceil(len(train_ds) / args.per_device_train_batch_size)
    steps_per_epoch = math.ceil(n_batches / args.gradient_accumulation_steps)
    total_steps = math.ceil(args.num_train_epochs * steps_per_epoch)
    steps = {"n_train": len(train_ds), "steps_per_epoch": steps_per_epoch, "total_steps": total_steps,
             "warmup_steps": args.get_warmup_steps(total_steps), "warmup_spec": 0.03}
    print(f"lr {lr:g}:", steps)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
    ckpts = sorted(glob.glob(f"{out}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    trainer.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
    assert trainer.state.max_steps == total_steps, (trainer.state.max_steps, total_steps)
    print(f"lr {lr:g}: 실제 전체 step {trainer.state.max_steps}, warmup {args.get_warmup_steps(trainer.state.max_steps)}")
    (out / "steps.json").write_text(json.dumps(steps, indent=1))
    (out / "log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
    (out / "done").write_text("ok")
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

## 설정 선택: 검증(2022) 200개 greedy 생성의 근거 검사 통과율(a·b·c), 동률이면 검증 손실이 낮은 쪽, 그다음 작은 학습률·적은 epoch

In [ ]:
from peft import PeftModel
results = []
for lr in LRS:
    out = LORA_DIR / f"r{G['r']}_lr{lr:g}"
    hist = json.loads((out / "log_history.json").read_text())
    evals = [h for h in hist if "eval_loss" in h]
    ckpts = sorted(glob.glob(f"{out}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    for ep, ck in enumerate(ckpts, start=1):
        path = out / f"val_gen_ep{ep}.jsonl"
        if len(dc.read_jsonl(path)) < len(val_in):
            base, tok_g = dc.load_student(deploy=False)
            model = PeftModel.from_pretrained(base, ck).eval()
            dc.student_generate(model, tok_g, prompt, val_in, batch_size=16, out_path=path, label=f"lr{lr:g} ep{ep}")
            del model, base; gc.collect(); torch.cuda.empty_cache()
        gens = {r["id"]: r["text"] for r in dc.read_jsonl(path)}
        checks = [grounding.check(gens[r["id"]], r["input"]) for r in val_in]
        results.append({"lr": lr, "epoch": ep, "checkpoint": ck, "val_pass": sum(c["pass"] for c in checks) / len(checks),
                        "val_format": sum(c["format"] for c in checks) / len(checks),
                        "val_loss": evals[ep - 1]["eval_loss"]})
chosen = sorted(results, key=lambda x: (-x["val_pass"], x["val_loss"], x["lr"], x["epoch"]))[0]
sel = {"grid": {k: v for k, v in G.items()}, "results": results, "chosen": chosen, "n_train": len(train_ds), "n_val": len(val_in),
       "smoke": SMOKE, "versions": PINS}
(LORA_DIR / "selection.json").write_text(json.dumps(sel, indent=1, default=str))
print(json.dumps(sel, indent=1, default=str))

In [ ]:
# 고른 어댑터만 lora/chosen/에 복사하고 해시를 남긴다
import shutil
dst = LORA_DIR / "chosen"
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(chosen["checkpoint"], dst, ignore=shutil.ignore_patterns("optimizer.pt", "scheduler.pt", "rng_state*", "trainer_state.json", "training_args.bin"))
print(dc.dir_hashes(dst))

## repo로 가져올 파일
- 본 실행(`SMOKE = False`)만 가져온다. `lora_smoke/`는 가져오지 않는다.
- `lora/selection.json`, 각 설정의 `log_history.json`·`steps.json`, `val_gen_ep*.jsonl` → `distill/lora/`
- `lora/chosen/`(adapter_config.json, adapter_model.safetensors) → `distill/lora/chosen/`
  - r 16, 모든 linear 대상이면 adapter 파라미터가 약 6,100만 개라 파일이 약 120MB(bf16)~250MB(fp32)다. GitHub의 파일당 100MB 제한을 넘으므로 push할 때는 Git LFS를 쓰거나 Drive에 두고 해시만 기록한다.

In [ ]:
# 끝: Drive에 다 쓴 뒤 런타임을 끈다(끝난 뒤 GPU가 켜진 채 컴퓨팅 단위를 쓰지 않게).
# 앞 셀에서 오류로 멈추면 이 셀은 실행되지 않는다. 그때는 직접 런타임 > 런타임 연결 해제 및 삭제.
AUTO_SHUTDOWN = True
if AUTO_SHUTDOWN:
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()